# 使用领域（私有）数据微调 ChatGLM3

生成带有 epoch 和 timestamp 的模型文件

In [1]:
import torch
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
print(torch.__config__.show(), torch.cuda.get_device_properties(0))

PyTorch built with:
  - GCC 9.3
  - C++ Version: 201703
  - Intel(R) oneAPI Math Kernel Library Version 2022.2-Product Build 20220804 for Intel(R) 64 architecture applications
  - Intel(R) MKL-DNN v3.3.6 (Git Hash 86e6af5974177e513fd3fee58425e1063e7f1361)
  - OpenMP 201511 (a.k.a. OpenMP 4.5)
  - LAPACK is enabled (usually provided by MKL)
  - NNPACK is enabled
  - CPU capability usage: AVX512
  - CUDA Runtime 12.1
  - NVCC architecture flags: -gencode;arch=compute_50,code=sm_50;-gencode;arch=compute_60,code=sm_60;-gencode;arch=compute_70,code=sm_70;-gencode;arch=compute_75,code=sm_75;-gencode;arch=compute_80,code=sm_80;-gencode;arch=compute_86,code=sm_86;-gencode;arch=compute_90,code=sm_90
  - CuDNN 8.9.2
  - Magma 2.6.1
  - Build settings: BLAS_INFO=mkl, BUILD_TYPE=Release, CUDA_VERSION=12.1, CUDNN_VERSION=8.9.2, CXX_COMPILER=/opt/rh/devtoolset-9/root/usr/bin/c++, CXX_FLAGS= -D_GLIBCXX_USE_CXX11_ABI=0 -fabi-version=11 -fvisibility-inlines-hidden -DUSE_PTHREADPOOL -DNDEBUG -DUSE_KINET

In [2]:
# 定义全局变量和参数
model_name_or_path = 'THUDM/chatglm3-6b'  # 模型ID或本地路径
# train_data_path = 'data/zhouyi_dataset_handmade.csv'    # 训练数据路径
train_data_path = 'data/zhouyi_dataset_20240118_163659.csv'    # 训练数据路径(批量生成数据集）
eval_data_path = None                     # 验证数据路径，如果没有则设置为None
seed = 8                                 # 随机种子
max_input_length = 128                    # 输入的最大长度
max_output_length = 32                   # 输出的最大长度
lora_rank = 16                             # LoRA秩
lora_alpha = 32                           # LoRA alpha值
lora_dropout = 0.05                       # LoRA Dropout率
prompt_text = ''                          # 所有数据前的指令文本

## 数据处理

In [3]:
from datasets import load_dataset

from datasets import load_dataset
dataset = load_dataset("yelp_review_full")

/root/miniconda3/envs/peft/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['label', 'text'],
        num_rows: 650000
    })
    test: Dataset({
        features: ['label', 'text'],
        num_rows: 50000
    })
})


In [5]:
from datasets import ClassLabel, Sequence
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)
    
    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
        elif isinstance(typ, Sequence) and isinstance(typ.feature, ClassLabel):
            df[column] = df[column].transform(lambda x: [typ.feature.names[i] for i in x])
    display(HTML(df.to_html()))

In [6]:
show_random_elements(dataset["train"], num_examples=5)

,label,text
0,1 star,"This is the WORST pharmacy I have ever used! Customer service is horrible, there is always a long wait and twice now I have dropped off a prescription and heard nothing for days about it being ready, I go to pick it up and they don't have it and have to order. Courtesy call? None. Central location to my house is the only reason why I ever used this pharmacy, from now on I will drive the extra mile! Steer clear!"
1,2 star,Frequent this cafe regularly as i receive comps from the casino. The food is overall pretty good for the price but the service seems to be increasingly worse. A simple dinner generally takes 90 minutes due to the poor/slow service. Poor management can only be to blame.
2,3 stars,"After a recommendation from Rand H. my hubby & I came here Sunday night. The restaurant was very clean & had a nice layout. When we were 1st seated a male server took our drink order, then throughout the meal a female server took our plates. Both were very sweet. What did surprise me though were the bathrooms. The sink area was actually super clean but the stalls & toilets were VERY dirty. I just washed my hands & waited till we got home to use the restroom. \n\nThere were quite a few buffet options to chose from, plus a salad bar & sushi station. The stand out items for us were: crab legs, mussels baked w/cheese on top, green beans in some sort of sauce, mini eggrolls, sausage w/mustard & salmon. For dessert we each had some tiny cakes (coconut & coffee flavored) & I got a piece of this apple turnover looking thing w/some vanilla ice cream on top. \n\nTotal including tip was about $30. Fair price for a buffet & we would probably come again if we were in the mood for it."
3,1 star,I don't know how this place got good reviews. We stopped by for lunch and waited 40 minutes and did not get our food. They were so disorganized sending plates to the wrong tables. Our plate was last at a table for ten minutes and then the waitress bought it to our table we refused to take the order. We left without eating and wasted an hour. Would not recommend. Don't go
4,4 stars,The only place in the Valley that I trust for an excellent pedicure. It's only $18 and it includes a hot rock massage. What more you could you ask for? I pass dozens of other nails salons on my 30 minutes drive here but I wouldn't think if going anywhere else.


In [7]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_name_or_path,
                                          trust_remote_code=True,
                                          revision='b098244')

/root/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [8]:
# tokenize_func 函数

def tokenize_func(example, tokenizer, ignore_label_id=-100):
    """
    对单个数据样本进行tokenize处理。

    参数:
    example (dict): 包含'content'和'summary'键的字典，代表训练数据的一个样本。
    tokenizer (transformers.PreTrainedTokenizer): 用于tokenize文本的tokenizer。
    ignore_label_id (int, optional): 在label中用于填充的忽略ID，默认为-100。

    返回:
    dict: 包含'tokenized_input_ids'和'labels'的字典，用于模型训练。
    """
    rating_map = ["极差", "较差", "中等", "较好", "极好"]
    # 构建问题文本
    question = prompt_text + example['text']
    if example.get('input', None) and example['input'].strip():
        question += f'\n{example["input"]}'

    # 构建答案文本
    answer = rating_map[example['label']]

    # 对问题和答案文本进行tokenize处理
    q_ids = tokenizer.encode(text=question, add_special_tokens=False)
    a_ids = tokenizer.encode(text=answer, add_special_tokens=False)

    # 如果tokenize后的长度超过最大长度限制，则进行截断
    if len(q_ids) > max_input_length - 2:  # 保留空间给gmask和bos标记
        q_ids = q_ids[:max_input_length - 2]
    if len(a_ids) > max_output_length - 1:  # 保留空间给eos标记
        a_ids = a_ids[:max_output_length - 1]

    # 构建模型的输入格式
    input_ids = tokenizer.build_inputs_with_special_tokens(q_ids, a_ids)
    question_length = len(q_ids) + 2  # 加上gmask和bos标记

    # 构建标签，对于问题部分的输入使用ignore_label_id进行填充
    labels = [ignore_label_id] * question_length + input_ids[question_length:]

    return {'input_ids': input_ids, 'labels': labels}


In [9]:
column_names = dataset['train'].column_names
tokenized_dataset = dataset.map(
    lambda example: tokenize_func(example, tokenizer),
    batched=False, 
    remove_columns=column_names
)
# tokenized_dataset = tokenized_dataset.shuffle(seed=seed)
# tokenized_dataset = tokenized_dataset.flatten_indices()

In [10]:
tokenized_dataset_train = tokenized_dataset['train'].shuffle(seed=42).select(range(3000))
tokenized_dataset_train = tokenized_dataset_train.flatten_indices()
tokenized_dataset_test = tokenized_dataset['test'].shuffle(seed=42).select(range(3000))
tokenized_dataset_test = tokenized_dataset_test.flatten_indices()
# show_random_elements(tokenized_dataset, num_examples=5)

,input_ids,labels
0,"[64790, 64792, 20849, 3690, 267, 1077, 1462, 332, 260, 5351, 1765, 30932, 12481, 3249, 14185, 658, 354, 12085, 560, 17877, 15168, 1096, 24180, 7726, 331, 260, 473, 275, 668, 470, 30930, 359, 12175, 658, 523, 849, 627, 26870, 14871, 540, 2408, 289, 267, 5890, 667, 14699, 30932, 12481, 3249, 323, 260, 7455, 623, 1234, 7896, 1626, 756, 260, 24533, 560, 498, 11524, 330, 1357, 611, 422, 434, 30930, 1310, 307, 726, 6672, 12481, 3249, 307, 401, 15736, 560, 307, 599, 628, 12710, 260, 1362, 30932, 3554, 4915, 293, 430, 915, 548, 30930, 6063, 307, 1153, 260, 3153, 6434, 1623, 30932, ...]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, ...]"
1,"[64790, 64792, 307, 1379, 434, 1234, 30930, 265, 30936, 1050, 289, 1967, 291, 6496, 350, 2531, 4654, 30930, 265, 30936, 726, 2077, 1132, 627, 362, 260, 2579, 18159, 30932, 832, 3389, 7177, 709, 941, 267, 1829, 323, 30930, 265, 1036, 350, 8306, 8243, 9288, 15796, 323, 7185, 30992, 265, 30936, 933, 2654, 941, 1829, 332, 260, 6768, 2586, 30930, 265, 1036, 280, 423, 8803, 383, 941, 1193, 30992, 30910, 41028, 2]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 30910, 41028, 2]"
2,"[64790, 64792, 316, 1251, 290, 451, 599, 8413, 985, 544, 3258, 293, 307, 1570, 267, 4524, 614, 14633, 30930, 701, 30916, 30974, 30916, 30936, 865, 431, 3587, 388, 267, 4524, 30932, 498, 1638, 30953, 30912, 893, 12134, 13211, 401, 1132, 289, 1527, 506, 30981, 30992, 4291, 30932, 260, 4683, 290, 12134, 13211, 401, 506, 30981, 30930, 577, 1638, 30953, 30912, 6755, 4101, 23993, 286, 400, 599, 2216, 1720, 291, 354, 30987, 7794, 434, 13211, 7332, 422, 2904, 28055, 3392, 30987, 17386, 267, 400, 6806, 7055, 422, 319, 5006, 28655, 30987, 4289, 4673, 2939, 11747, 356, 434, 13961, 12134, 13211, 30932, ...]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, ...]"
3,"[64790, 64792, 353, 11078, 30917, 293, 307, 3359, 289, 1961, 434, 1234, 528, 331, 260, 3853, 30910, 21819, 28278, 3507, 6659, 30930, 7178, 881, 267, 5100, 401, 11434, 400, 343, 379, 1818, 919, 472, 291, 267, 3997, 290, 267, 6659, 30932, 8634, 8634, 401, 1114, 5992, 30930, 577, 3205, 496, 260, 3391, 289, 7155, 537, 6839, 293, 3397, 290, 1753, 359, 30937, 261, 20211, 401, 331, 649, 577, 30953, 30917, 260, 3528, 30932, 25202, 1234, 356, 260, 4160, 8004, 30930, 307, 2628, 343, 30930, 701, 30916, 30974, 30916, 2266, 3549, 267, 26287, 1486, 30119, 6693, 1893, 30917, 30932, 267, 23469, ...]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100

In [11]:
import torch
from typing import List, Dict, Optional

# DataCollatorForChatGLM 类
class DataCollatorForChatGLM:
    """
    用于处理批量数据的DataCollator，尤其是在使用 ChatGLM 模型时。

    该类负责将多个数据样本（tokenized input）合并为一个批量，并在必要时进行填充(padding)。

    属性:
    pad_token_id (int): 用于填充(padding)的token ID。
    max_length (int): 单个批量数据的最大长度限制。
    ignore_label_id (int): 在标签中用于填充的ID。
    """

    def __init__(self, pad_token_id: int, max_length: int = 2048, ignore_label_id: int = -100):
        """
        初始化DataCollator。

        参数:
        pad_token_id (int): 用于填充(padding)的token ID。
        max_length (int): 单个批量数据的最大长度限制。
        ignore_label_id (int): 在标签中用于填充的ID，默认为-100。
        """
        self.pad_token_id = pad_token_id
        self.ignore_label_id = ignore_label_id
        self.max_length = max_length

    def __call__(self, batch_data: List[Dict[str, List]]) -> Dict[str, torch.Tensor]:
        """
        处理批量数据。

        参数:
        batch_data (List[Dict[str, List]]): 包含多个样本的字典列表。

        返回:
        Dict[str, torch.Tensor]: 包含处理后的批量数据的字典。
        """
        # 计算批量中每个样本的长度
        len_list = [len(d['input_ids']) for d in batch_data]
        batch_max_len = max(len_list)  # 找到最长的样本长度

        input_ids, labels = [], []
        for len_of_d, d in sorted(zip(len_list, batch_data), key=lambda x: -x[0]):
            pad_len = batch_max_len - len_of_d  # 计算需要填充的长度
            # 添加填充，并确保数据长度不超过最大长度限制
            ids = d['input_ids'] + [self.pad_token_id] * pad_len
            label = d['labels'] + [self.ignore_label_id] * pad_len
            if batch_max_len > self.max_length:
                ids = ids[:self.max_length]
                label = label[:self.max_length]
            input_ids.append(torch.LongTensor(ids))
            labels.append(torch.LongTensor(label))

        # 将处理后的数据堆叠成一个tensor
        input_ids = torch.stack(input_ids)
        labels = torch.stack(labels)

        return {'input_ids': input_ids, 'labels': labels}


In [12]:
# 准备数据整理器
data_collator = DataCollatorForChatGLM(pad_token_id=tokenizer.pad_token_id)

## 加载模型

In [13]:
from transformers import AutoModel, BitsAndBytesConfig

_compute_dtype_map = {
    'fp32': torch.float32,
    'fp16': torch.float16,
    'bf16': torch.bfloat16
}

# QLoRA 量化配置
q_config = BitsAndBytesConfig(load_in_4bit=True,
                              bnb_4bit_quant_type='nf4',
                              bnb_4bit_use_double_quant=True,
                              bnb_4bit_compute_dtype=_compute_dtype_map['bf16'])
# 加载量化后模型
model = AutoModel.from_pretrained(model_name_or_path,
                                  quantization_config=q_config,
                                  device_map='auto',
                                  trust_remote_code=True,
                                  revision='b098244')

model.supports_gradient_checkpointing = True
model.gradient_checkpointing_enable()
model.enable_input_require_grads()

model.config.use_cache = True

/root/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 7/7 [00:04<00:00,  1.55it/s]
/root/miniconda3/envs/peft/lib/python3.10/site-packages/huggingface_hub/file_download.py:797: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` i

In [14]:
from peft import TaskType, LoraConfig, get_peft_model, prepare_model_for_kbit_training
from peft.utils import TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING

kbit_model = prepare_model_for_kbit_training(model)
target_modules = TRANSFORMERS_MODELS_TO_LORA_TARGET_MODULES_MAPPING['chatglm']

You are using an old version of the checkpointing format that is deprecated (We will also silently ignore `gradient_checkpointing_kwargs` in case you passed it).Please update to the new format on your modeling file. To use the new format, you need to completely remove the definition of the method `_set_gradient_checkpointing` in your model.


In [15]:
lora_config = LoraConfig(
    target_modules=target_modules,
    r=lora_rank,
    lora_alpha=lora_alpha,
    lora_dropout=lora_dropout,
    bias='none',
    inference_mode=False,
    task_type=TaskType.CAUSAL_LM
)

In [16]:
qlora_model = get_peft_model(kbit_model, lora_config)
qlora_model.print_trainable_parameters()

trainable params: 3,899,392 || all params: 6,247,483,392 || trainable%: 0.06241540401681151


### QLoRA 微调模型

In [17]:
import datetime

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

train_epochs = 3
output_dir = f"models/{model_name_or_path}-yelp-epoch{train_epochs}-{timestamp}"

In [18]:
tokenized_dataset

Dataset({
    features: ['input_ids', 'labels'],
    num_rows: 3000
})

In [19]:
show_random_elements(tokenized_dataset, num_examples=5)

,input_ids,labels
0,"[64790, 64792, 30910, 30940, 3829, 16381, 30930, 666, 17673, 16931, 323, 14415, 30930, 353, 3969, 480, 430, 683, 709, 289, 480, 537, 1672, 30930, 2095, 1095, 620, 601, 8690, 1158, 293, 430, 2814, 30930, 577, 1933, 1194, 3969, 289, 6653, 552, 1514, 541, 30930, 13771, 1982, 289, 2036, 434, 12695, 8467, 290, 260, 3378, 30930, 30910, 55023, 55342, 2]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, 30910, 55023, 55342, 2]"
1,"[64790, 64792, 3438, 7238, 272, 3262, 293, 5204, 659, 260, 13399, 6120, 30979, 13515, 30932, 552, 726, 14950, 2073, 496, 4758, 30930, 265, 20060, 267, 1829, 323, 430, 291, 286, 860, 30932, 354, 4402, 953, 5553, 27570, 3984, 5905, 17492, 272, 30930, 265, 30936, 7458, 284, 2569, 8084, 7739, 280, 19362, 793, 260, 9415, 290, 5399, 30934, 1702, 30923, 17492, 272, 356, 18468, 30930, 265, 1036, 18468, 323, 2430, 659, 267, 18468, 343, 2114, 356, 10390, 17492, 272, 30930, 265, 8570, 1597, 30932, 354, 401, 15544, 30932, 6714, 30932, 293, 11473, 1290, 30930, 265, 1036, 27017, 773, 293, 13384, 383, ...]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, ...]"
2,"[64790, 64792, 307, 30953, 318, 628, 467, 1955, 283, 289, 863, 280, 18274, 1325, 3923, 985, 540, 379, 10015, 528, 6204, 25319, 336, 456, 1059, 1788, 30930, 657, 1390, 627, 30910, 30972, 30970, 2604, 1000, 267, 2476, 2077, 793, 267, 1234, 401, 3589, 11561, 560, 356, 260, 1699, 30910, 30966, 30940, 705, 2990, 289, 2505, 291, 30930, 16308, 344, 431, 289, 634, 260, 3583, 30953, 30917, 2975, 400, 1755, 344, 30953, 602, 742, 634, 30910, 30939, 30967, 30972, 290, 267, 1610, 17844, 30930, 307, 401, 430, 3879, 291, 343, 21282, 30930, 307, 401, 629, 430, 3879, 291, 6097, 291, 1699, ...]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, ...]"
3,"[64790, 64792, 307, 457, 30953, 30912, 1105, 1775, 878, 1257, 565, 434, 1234, 30930, 353, 23844, 293, 6705, 30922, 1145, 383, 844, 11473, 759, 293, 808, 12786, 293, 629, 844, 18780, 30930, 353, 1077, 1627, 307, 431, 289, 1105, 323, 523, 431, 830, 30519, 531, 3775, 1548, 30930, 657, 431, 7458, 23844, 428, 3365, 309, 16984, 1436, 1851, 1703, 293, 523, 431, 1366, 628, 331, 622, 293, 680, 362, 379, 7458, 30930, 16524, 1661, 761, 814, 307, 1112, 431, 687, 12490, 30992, 1118, 430, 742, 5531, 289, 634, 451, 597, 10307, 331, 622, 498, 30992, 523, 862, 12772, 291, 21870, ...]","[-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -10

In [20]:
# qlora_model = qlora_model.to("cuda:0")
from transformers import TrainingArguments, Trainer

training_args = TrainingArguments(
    output_dir=output_dir,                            # 输出目录
    per_device_train_batch_size=8,                     # 每个设备的训练批量大小
    gradient_accumulation_steps=1,                     # 梯度累积步数
    learning_rate=1e-3,                                # 学习率
    num_train_epochs=train_epochs,                     # 训练轮数
    lr_scheduler_type="linear",                        # 学习率调度器类型
    warmup_ratio=0.1,                                  # 预热比例
    logging_steps=1,                                 # 日志记录步数
    save_strategy="steps",                             # 模型保存策略
    save_steps=10,                                    # 模型保存步数
    optim="adamw_torch",                               # 优化器类型
    fp16=True,                                        # 是否使用混合精度训练
)

trainer = Trainer(
        model=qlora_model,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=data_collator
    )

In [21]:
# trainer = Trainer(
#         model=qlora_model,
#         args=training_args,
#         train_dataset=tokenized_dataset,
#         data_collator=data_collator
#     )

In [ ]:
trainer.train()

`use_cache=True` is incompatible with gradient checkpointing. Setting `use_cache=False`...
/root/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(


Step,Training Loss
1,11.508400
2,10.149800
3,10.377600
4,11.381800
5,11.791100
6,11.071900
7,9.523400
8,10.256800
9,12.037300
10,11.777900


/root/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/root/miniconda3/envs/peft/lib/python3.10/site-packages/torch/utils/checkpoint.py:464: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.4 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  warnings.warn(
/root/miniconda3/envs/peft/lib/python3.10/site-pac

In [20]:
trainer.model.save_pretrained(output_dir)